# 01_Agent详解

**来源:**
- [LangChain Docs — Deep Agents SDK 概述](https://docs.langchain.com/oss/python/deepagents/overview)
- [LangChain Docs — Agents](https://docs.langchain.com/oss/python/deepagents/agents)
- [LangChain Docs — Harness capabilities](https://docs.langchain.com/oss/python/deepagents/harness)
- [LangChain Docs — Models](https://docs.langchain.com/oss/python/deepagents/models)
- [LangChain Docs — Deep Agents Code 概述](https://docs.langchain.com/docs/deep-agents/code/overview)

本 Notebook 全面讲解 `create_deep_agent` 的所有参数、工具定义、系统提示、结构化输出、调用方式、流式、中间件、沙箱后端、技能和子 Agent 等内容。

## 目录

1. [一句话创建 Agent](#1-一句话创建-agent)
2. [create_deep_agent 参数详解](#2-create_deep_agent-参数详解)
3. [工具定义](#3-工具定义)
4. [System Prompt 与 Memory](#4-system-prompt-与-memory)
5. [结构化输出 (Response Format)](#5-结构化输出-response-format)
6. [Invocation 调用方式](#6-invocation-调用方式)
7. [Streaming 流式输出](#7-streaming-流式输出)
8. [中间件 (Middleware)](#8-中间件-middleware)
9. [沙箱后端 (Sandbox Backend)](#9-沙箱后端-sandbox-backend)
10. [技能 (Skills)](#10-技能-skills)
11. [子 Agent (Subagents)](#11-子-agent-subagents)
12. [Harness Profile](#12-harness-profile)

**参考链接**
- [Deep Agents SDK 文档](https://docs.langchain.com/oss/python/deepagents)
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)
- [LangChain 聊天模型集成](https://docs.langchain.com/oss/python/integrations/chat)

## 1. 一句话创建 Agent

Deep Agents 用最少的样板代码就能创建一个可用 Agent：

In [ ]:
# 安装: pip install -qU deepagents langchain langgraph

from deepagents import create_deep_agent

# 定义一个简单工具（函数即可，@tool 装饰器可选）
def get_weather(city: str) -> str:
    """获取城市天气信息"""
    return f"{city} 的天气是晴天，温度 25°C！"

# 一行创建 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",  # 模型标识: provider:model
    tools=[get_weather],                     # 工具列表
    system_prompt="你是一个有用的天气助手。",  # 系统提示
)

# 调用 Agent
result = agent.invoke({
    "messages": [{"role": "user", "content": "旧金山的天气怎么样？"}]
})
print(result["messages"][-1].content)

## 2. create_deep_agent 参数详解

`create_deep_agent` 是创建 Agent 的核心入口函数。其参数如下：

| 参数 | 类型 | 必需 | 说明 |
|------|------|------|------|
| `model` | `str \| BaseChatModel` | ✅ | 模型标识符（如 `"anthropic:claude-sonnet-4-6"`）或 LangChain ChatModel 实例 |
| `tools` | `list[Callable \| BaseTool]` | ❌ | Agent 可调用的工具列表 |
| `system_prompt` | `str` | ❌ | 系统提示词 |
| `subagents` | `list[dict \| CompiledSubAgent]` | ❌ | 子 Agent 配置列表 |
| `middleware` | `list[Middleware]` | ❌ | 中间件列表 |
| `model_params` | `dict` | ❌ | 模型额外参数（如 temperature、max_tokens） |
| `sandbox` | `Backend` | ❌ | 沙箱后端，用于执行 Shell 命令 |
| `checkpointer` | `BaseCheckpointSaver` | ❌ | 检查点保存器，用于多轮对话和中断恢复 |
| `memory` | `list[str]` | ❌ | 持久化记忆文件路径列表 |
| `skills` | `list[str]` | ❌ | 技能源路径 |
| `permissions` | `list[FilesystemPermission]` | ❌ | 文件系统权限规则 |
| `response_format` | `ResponseFormat` | ❌ | 结构化输出模式 |
| `interrupt_on` | `dict` | ❌ | 为特定工具配置人工审批 |
| `harness_profile` | `str \| HarnessProfile` | ❌ | 覆盖默认的 Harness 配置 |
| `name` | `str` | ❌ | Agent 名称，用于追踪和日志 |
| `tracing_name` | `str` | ❌ | LangSmith 追踪名称 |

### 2.1 模型标识格式

格式: `provider:model`

```
google_genai:gemini-3.5-flash
openai:gpt-5.4
anthropic:claude-sonnet-4-6
baseten:zai-org/GLM-5.1
```

| 部分 | 含义 |
|------|------|
| `provider` | LangChain 集成名，即 `init_chat_model` 的 `model_provider` 参数 |
| `model` | Provider 侧模型标识符 |

### 2.2 支持的 Provider

| Provider | 字符串格式示例 | 安装包 |
|----------|--------------|--------|
| Google | `google_genai:gemini-3.5-flash` | `langchain-google-genai` |
| OpenAI | `openai:gpt-5.4` | `langchain-openai` |
| Anthropic | `anthropic:claude-sonnet-4-6` | `langchain-anthropic` |
| OpenRouter | `openrouter:anthropic/claude-sonnet-4-6` | `langchain-openrouter` |
| Fireworks | `fireworks:accounts/fireworks/models/qwen3p5-397b-a17b` | `langchain-fireworks` |
| Baseten | `baseten:zai-org/GLM-5` | `langchain-baseten` |
| Ollama | `ollama:qwen3:4b` | `langchain-ollama` |
| Groq | `groq:mixtral-8x7b-32768` | `langchain-groq` |
| DeepSeek | `deepseek:deepseek-chat` | `langchain-deepseek` |
| Azure OpenAI | `azure:gpt-4o` | `langchain-openai` |
| AWS Bedrock | `bedrock:anthropic.claude-sonnet-4-6` | `langchain-aws` |

### 2.3 使用完整的 Agent 配置

以下展示一个包含大多数参数的完整配置：

In [ ]:
# 完整配置示例
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[get_weather, search_database],  # 自定义工具
    system_prompt="你是一个专业的数据分析师。",
    model_params={"temperature": 0.2, "max_tokens": 4096},
    sandbox=LocalShellBackend(),           # 启用 execute 工具
    checkpointer=MemorySaver(),            # 支持多轮对话
    memory=["project/AGENTS.md"],          # 持久化记忆
    permissions=[],                        # 文件系统权限
    middleware=[],                         # 中间件
    subagents=[],                          # 子 Agent
    name="my-data-agent",
)

## 3. 工具定义

Deep Agents 支持多种工具定义方式：

In [ ]:
# 方式1: 纯函数（自动推断 schema）
def search_db(query: str, limit: int = 10) -> list:
    """搜索数据库，返回最多 limit 条结果"""
    return [f"结果{i}" for i in range(limit)]


# 方式2: @tool 装饰器（更明确的控制）
from langchain.tools import tool

@tool
def fetch_weather(city: str, units: str = "celsius") -> str:
    """获取指定城市的天气信息。

    参数:
        city: 城市名称
        units: 温度单位（celsius/fahrenheit）
    """
    return f"{city}: 25°{units[0].upper()}"


# 方式3: BaseTool 子类（完整控制）
from langchain.tools import BaseTool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(description="城市名称")
    units: str = Field(default="celsius", description="温度单位")

class WeatherTool(BaseTool):
    name: str = "weather"
    description: str = "获取城市天气信息"
    args_schema: type = WeatherInput

    def _run(self, city: str, units: str = "celsius") -> str:
        return f"{city}: 25°{units[0].upper()}"


# 方式4: StructuredTool（快捷方式）
from langchain.tools import StructuredTool

def calculate(x: float, y: float, op: str = "add") -> float:
    """执行数学运算"""
    if op == "add":
        return x + y
    elif op == "multiply":
        return x * y
    return x - y

calc_tool = StructuredTool.from_function(
    func=calculate,
    name="calculator",
    description="执行数学运算（加/减/乘）",
)

### 3.1 内置工具

Deep Agents 自动提供以下内置文件系统工具（通过 Harness 加载）：

| 工具 | 说明 | 人工审批 |
|------|------|---------|
| `ls` | 列出目录中的文件 | - |
| `read_file` | 读取文件内容（支持 offset/limit） | - |
| `write_file` | 创建新文件 | 需要 |
| `edit_file` | 精确字符串替换编辑 | 需要 |
| `glob` | 查找匹配模式的文件（如 `**/*.py`） | - |
| `grep` | 搜索文件内容 | - |
| `execute` | 运行 Shell 命令（仅沙箱后端可用）| 需要 |
| `web_search` | 使用 Tavily 搜索网页 | 需要 |
| `fetch_url` | 获取网页并转为 Markdown | 需要 |
| `task` | 将工作委派给子 Agent | 需要 |
| `ask_user` | 向用户提问 | - |
| `write_todos` | 创建和管理任务列表 | - |
| `compact_conversation` | 压缩对话历史 | 混合 |

### 3.2 排除默认工具

通过 `HarnessProfile` 可排除不需要的内置工具：

In [ ]:
from deepagents import HarnessProfile, register_harness_profile

# 注册一个配置，隐藏默认文件系统工具
register_harness_profile(
    "anthropic:claude-sonnet-4-6",
    HarnessProfile(
        excluded_tools=frozenset({
            "ls", "read_file", "write_file", "edit_file",
            "glob", "grep"
        }),
    ),
)

# 之后创建的 Agent 将不包含排除的工具
agent_no_fs = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[fetch_weather],
)

### 3.3 文件系统权限 (Permissions)

声明式权限规则，控制 Agent 可读写哪些路径：

| 参数 | 说明 |
|------|------|
| `action` | `"read"` 或 `"write"` |
| `path` | Glob 模式匹配路径 |
| `mode` | `"allow"` 或 `"deny"` |

- **首条匹配规则生效**（first-match-wins）
- 若无规则匹配，操作默认允许

In [ ]:
from deepagents.permissions import FilesystemPermission

permissions = [
    # 允许读取 /data 目录
    FilesystemPermission(action="read", path="/data/**", mode="allow"),
    # 拒绝读取敏感文件
    FilesystemPermission(action="read", path="/data/secrets/**", mode="deny"),
    # 只允许写入 /output 目录
    FilesystemPermission(action="write", path="/output/**", mode="allow"),
    FilesystemPermission(action="write", path="**", mode="deny"),
]

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    permissions=permissions,
)

## 4. System Prompt 与 Memory

### 4.1 System Prompt 层级

System Prompt 的组成（按加载顺序）：

```text
包基础提示（始终加载）
+ ~/.deepagents/{agent}/AGENTS.md（追加）
+ .deepagents/AGENTS.md（追加）
+ AGENTS.md（项目根目录，追加）
+ 传入的 system_prompt 参数
```

### 4.2 Memory 记忆

Memory 是持久化的额外上下文，通过 `memory` 参数传入文件路径列表：

In [ ]:
# 为 Agent 配置记忆文件
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[fetch_weather],
    memory=["project/AGENTS.md"],  # 持久化记忆
)

# 记忆文件内容示例:
# ## 编码风格
# - 使用 snake_case 命名
# - 包含类型注解
# - 每个函数都写 docstring

## 5. 结构化输出 (Response Format)

Deep Agents 支持通过 `response_format` 参数约束 Agent 输出为结构化数据：

In [ ]:
from pydantic import BaseModel

# 定义输出结构
class WeatherReport(BaseModel):
    city: str
    temperature: float
    units: str
    condition: str

# 创建带结构化输出的 Agent
agent = create_deep_agent(
    model="openai:gpt-5.4",
    tools=[fetch_weather],
    response_format=WeatherReport,  # 输出格式约束
)

# 调用后，结果将被解析为 WeatherReport 对象
result = agent.invoke({
    "messages": [{"role": "user", "content": "东京天气怎么样？"}]
})
# result["messages"][-1] 将包含结构化数据

## 6. Invocation 调用方式

### 6.1 invoke（同步调用）

最直接的调用方式，等待 Agent 完成所有步骤后返回最终结果。

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "旧金山天气怎么样？"}]
})

# 获取最终回答
print(result["messages"][-1].content)

# 获取完整对话历史
for msg in result["messages"]:
    print(f"[{msg.type}]: {msg.content[:100]}...")

### 6.2 ainvoke（异步调用）

In [ ]:
# 异步调用
import asyncio

async def run_agent():
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": "纽约天气怎么样？"}]
    })
    return result["messages"][-1].content

result = asyncio.run(run_agent())
print(result)

## 7. Streaming 流式输出

### 7.1 stream（基本流式）

使用 `agent.stream()` 获取逐步更新：

In [ ]:
# 流式输出更新
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "讲讲量子计算"}]},
    stream_mode="updates",  # updates / messages / custom
):
    print(chunk)

### 7.2 stream_events（事件流 API）

Deep Agents v0.6 引入的类型化投影 API：

In [ ]:
# 使用 stream_events API
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "写一首关于编程的诗"}]},
    version="v3",
)

# 获取消息流
for message in stream.messages:
    print(f"[消息] {message.text}")

# 获取工具调用
for call in stream.tool_calls:
    print(f"[工具] {call.tool_name}({call.input})")

# 获取子 Agent
for subagent in stream.subagents:
    print(f"[子Agent] {subagent.name}: {subagent.status}")

更多流式细节参见《运行时与流式/01_流式与事件流.ipynb》。

## 8. 中间件 (Middleware)

Middleware 是运行在 Agent 调用链中的钩子，用于日志记录、监控、修改行为等。

| 中间件 | 说明 |
|--------|------|
| `@dynamic_prompt` | 动态修改系统提示 |
| `@wrap_model_call` | 包装模型调用（如添加重试、日志） |
| 内置中间件 | 速率限制、超时、令牌计数等 |

更多细节参见《中间件/01_中间件.ipynb》。

## 9. 沙箱后端 (Sandbox Backend)

沙箱后端为 Agent 提供隔离的代码执行环境。Agent 的 `execute` 工具需要沙箱后端才能工作。

| 提供商 | 安装命令 | 凭据 |
|--------|---------|------|
| `LocalShellBackend` | 默认内置 | 无 |
| LangSmith | 默认内置 | `LANGSMITH_API_KEY` |
| Daytona | `langchain-daytona` | `DAYTONA_API_KEY` |
| Modal | `langchain-modal` | Modal 设置 |
| Runloop | `langchain-runloop` | `RUNLOOP_API_KEY` |
| AgentCore | `langchain-agentcore-codeinterpreter` | AWS 凭据 |

In [ ]:
from deepagents.backends import LocalShellBackend

# 使用本地 Shell 后端（在本地执行命令）
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[fetch_weather],
    sandbox=LocalShellBackend(),  # 启用 execute 工具
)

## 10. 技能 (Skills)

Skills 提供专业化工作流和领域知识。遵循 **Agent Skills 标准**。

每个 Skill 是一个目录，包含 `SKILL.md` 文件：

```text
skills/
└── code-review/
    └── SKILL.md
```

### 技能发现路径（优先级从低到高）

```text
1. ~/.deepagents/{agent}/skills/     — 用户 Deep Agents
2. ~/.agents/skills/                — 用户工具无关
3. .deepagents/skills/              — 项目 Deep Agents
4. .agents/skills/                  — 项目工具无关（最高）
```

### 调用技能

```bash
# 会话中调用
/skill:code-review

# 启动时加载
dcode --skill code-review
```

### 在 SDK 中使用技能

In [ ]:
# 通过 skills 参数加载技能
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    skills=["./skills/code-review"],  # 技能源路径
)

## 11. 子 Agent (Subagents)

子 Agent 解决**上下文膨胀问题**。当 Agent 使用输出较大的工具（网络搜索、文件读取）时，上下文窗口很快被中间结果填满。子 Agent 隔离这些细节工作——主 Agent 只接收最终结果。

### 何时使用子 Agent

- ✅ 会混乱主 Agent 上下文的多步骤任务
- ✅ 需要自定义指令或工具的专业领域
- ✅ 需要不同模型能力的任务
- ✅ 希望主 Agent 专注于高层协调

### 子 Agent 配置字段

| 字段 | 类型 | 必填 | 说明 |
|------|------|------|------|
| `name` | str | ✅ | 唯一标识符 |
| `description` | str | ✅ | 功能描述，触发委托的依据 |
| `system_prompt` | str | ✅ | 子 Agent 的指令 |
| `tools` | list[Callable] | ❌ | 子 Agent 可用的工具 |
| `model` | str \| BaseChatModel | ❌ | 覆盖主 Agent 的模型 |
| `middleware` | list[Middleware] | ❌ | 额外中间件 |
| `interrupt_on` | dict | ❌ | 人工审批配置 |
| `skills` | list[str] | ❌ | 技能源路径 |
| `response_format` | ResponseFormat | ❌ | 结构化输出模式 |
| `permissions` | list[FilesystemPermission] | ❌ | 文件系统权限 |

In [ ]:
from langchain.tools import tool

@tool
def internet_search(query: str, max_results: int = 5) -> str:
    """执行网络搜索"""
    return f"关于 '{query}' 的搜索结果（共 {max_results} 条）..."


# 定义子 Agent（字典方式）
research_subagent = {
    "name": "research-agent",
    "description": "用于深入调研问题",
    "system_prompt": "你是一个优秀的研究员，善于深入挖掘信息。",
    "tools": [internet_search],
    "model": "google_genai:gemini-3.5-flash",  # 可选覆盖模型
}

# 创建带子 Agent 的深度 Agent
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    subagents=[research_subagent],
)

# 当用户要求研究时，主 Agent 会自动委托给 research-agent

### 11.1 CompiledSubAgent 方式

使用 `CompiledSubAgent` 类可以更精细地控制子 Agent：

In [ ]:
from deepagents import CompiledSubAgent

# 使用 CompiledSubAgent 方式
subagent = CompiledSubAgent(
    name="writer",
    description="创意写作",
    system_prompt="你是一个有创意的作家。",
    model="openai:gpt-5.4",
)

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    subagents=[subagent],
)

### 11.2 子 Agent 流式

启用 `subgraphs=True` 后，可以从主 Agent 和子 Agent 中实时流式输出更新：

In [ ]:
# 子 Agent 流式输出
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "研究量子计算"}]},
    stream_mode="updates",
    subgraphs=True,  # 启用子图流式
    version="v2",
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            # 子 Agent 事件
            print(f"[子Agent: {chunk['ns']}] {chunk['data']}")
        else:
            # 主 Agent 事件
            print(f"[主Agent] {chunk['data']}")

## 12. Harness Profile

Harness Profile 允许将每个模型的配置打包为可复用的捆绑包，统一管理工具集、中间件、权限等。

In [ ]:
from deepagents import HarnessProfile, register_harness_profile

# 创建自定义 Profile
my_profile = HarnessProfile(
    excluded_tools=frozenset({"web_search", "fetch_url"}),
    general_purpose_subagent_enabled=False,
    sandbox_enabled=True,
)

register_harness_profile("my-custom-profile", my_profile)

# 使用自定义 Profile
agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    harness_profile="my-custom-profile",
)

---

**参考链接**
- [Deep Agents SDK 文档](https://docs.langchain.com/oss/python/deepagents)
- [Deep Agents Agents 文档](https://docs.langchain.com/oss/python/deepagents/agents)
- [Harness 能力文档](https://docs.langchain.com/oss/python/deepagents/harness)
- [LangChain 聊天模型集成](https://docs.langchain.com/oss/python/integrations/chat)
- [Deep Agents Code 文档](https://docs.langchain.com/docs/deep-agents/code/overview)
- [LangGraph 流式文档](https://langchain-ai.github.io/langgraph/how-tos/stream-subgraphs/)
- [Agent Skills 标准](https://github.com/langchain-ai/agent-skills-spec)